# 03 — Ein TCOCNN trainieren und Vorhersagen verstehen

Ein kompletter Zyklus mit vier Subsensoren geht hinein; genau eine Gaskonzentration kommt heraus. Keine Hyperparameteroptimierung.


In [ ]:
from pathlib import Path
import sys,h5py,numpy as np,matplotlib.pyplot as plt
ROOT=next(p for p in (Path.cwd().resolve(),*Path.cwd().resolve().parents) if (p/'Networks'/'TCOCNNv3.py').exists())
sys.path.insert(0,str(ROOT/'Networks'))
sys.path.insert(0,str(ROOT/'Evaluation Seminar'/'Day_03'))
from TCOCNNv3 import TCOCNNv3Class
from day3_utils import prepare_model_data,train_experiment,regression_metrics,plot_comparison,show_results
with h5py.File(ROOT/'Data'/'fullData.mat','r') as f:
    AVAILABLE_GASES=sorted(k for k in f['targets_train'] if k not in {'range','calibration'})
print('Verfügbare Zielgase:',AVAILABLE_GASES)
GAS='acetone'
splits,scaler=prepare_model_data(GAS)
print('Input:',splits['train']['X_z'].shape,'= Zyklen × 4 Subsensoren × 1440 × 1')
print('Output:',splits['train']['y_z'].shape,'= eine Konzentration pro Zyklus')


## Modell sichtbar aufbauen

Die Parameter stehen direkt hier; anschließend wird die echte Architektur ausgegeben.


In [ ]:
PARAMS=dict(n_filter=24,section_depth=3,convs_per_block=3,channel_growth=16,kernel=9,stride=4,
 num_neurons=128,drop_out=0.0,initial_learning_rate=3e-4,residual=True,batch_size=64)
preview=TCOCNNv3Class(splits['train']['X_z'].shape[1:],1,regression=True)
preview.build_net(PARAMS)
print(preview.model)
print('Trainierbare Parameter:',sum(p.numel() for p in preview.model.parameters() if p.requires_grad))
model,info=train_experiment(splits,PARAMS,epochs=100,seed=42,model_class=TCOCNNv3Class)


## Lernkurve

Linear und logarithmisch, jeweils mit Grid und markiertem Checkpoint.


In [ ]:
h=model.history.history; e=np.arange(1,len(h['loss'])+1)
fig,axes=plt.subplots(1,2,figsize=(14,4))
for ax in axes:
    ax.plot(e,h['loss'],label='Training'); ax.plot(e,h['val_loss'],label='Validierung')
    ax.axvline(info['best_epoch'],color='black',ls=':',label='bester Checkpoint')
    ax.set(xlabel='Epoche',ylabel='MSE'); ax.grid(True,which='both',alpha=.3); ax.legend()
axes[0].set_title('Lernkurve – linear')
axes[1].set(title='Lernkurve – logarithmisch',yscale='log')
plt.show()


## Output in ppb

Die Modellwerte werden zurück in die physikalische Einheit transformiert.


In [ ]:
rows=[]
for name in ['val','test','test_extra']:
    truth=splits[name]['y']
    pred=model.predict(splits[name]['X_z']).ravel()*scaler['y_std']+scaler['y_mean']
    rows.append({'Split':name,**regression_metrics(truth,pred)})
    plot_comparison(truth,{f'TCOCNN ({GAS})':pred},f'{name}: Inputzyklus → Konzentration')
show_results(rows)
